# Datamine CHANNL3D Process Example

This notebook demonstrates how to configure and run the **`channl3d`** process wrapper in `dmstudio`.

## Process Description

## CHANNL3D

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

CHANNL3D converts a set of Channel Sample data and Channel Survey data into a 'desurveyed' Static Drillholes file in which each sample is identified by its location (X,Y,Z coordinates) and direction (azimuth and dip) in space. This is the same format as created by the [HOLES3D](<holes3d.md>) process for drillhole data.

The input data consists of one survey file and one or more sample files. If two or more Channel Sample files are specified (maximum 6) then they are merged so that all divisions of all sets of samples are maintained. A typical use of merging is to add an assay file to a lithology file, where the lithology intervals do not match the assay sample boundaries. Another example is to add absent data samples into channels by making the second file a single record per channel defining the total channel length.

The output from the CHANNL3D process is in the standard Drillhole format which is used extensively elsewhere in Studio. For example, the channels can be viewed interactively, composited along the channel, and used to interpolate grades into a block model.

### Input Files:

* **survpts** (Point Data):
  Survey data point file containing channel identifier and 3D coordinates. Expects fields
  BHID, XPT, YPT, ZPT. The points must be sorted by location along the channel.
  Required=Yes

* **sample1** (Downhole Sample):
  First sample data file. This file is compulsory and must include fields BHID, FROM, and
  TO. It will probably also include at least one sample attribute field, such as grade or
  lithology.
  Required=Yes

* **sample2** (Downhole Sample):
  Five optional sample data files containing BHID, FROM and TO, and probably at least one
  sample attribute field.
  Required=No

* **sample3** (Downhole Sample):
  Five optional sample data files containing BHID, FROM and TO, and probably at least one
  sample attribute field.
  Required=No

* **sample4** (Downhole Sample):
  Five optional sample data files containing BHID, FROM and TO, and probably at least one
  sample attribute field.
  Required=No

* **sample5** (Downhole Sample):
  Five optional sample data files containing BHID, FROM and TO, and probably at least one
  sample attribute field.
  Required=No

* **sample6** (Downhole Sample):
  Five optional sample data files containing BHID, FROM and TO, and probably at least one
  sample attribute field.
  Required=No

### Output Files:

* **out** (Yes):
  Drillhole
  Required=No

* **chansmry** (No):
  Table
  Required=No

* **errors** (No):
  Table
  Required=No

### Fields:

* **bhid** (Any : SURVPTS SAMPLE1):
  Channel identifier
  Default=BHID
  Required=No

* **xpt** (Numeric : SURVPTS):
  X coordinate of survey point
  Default=XPT
  Required=No

* **ypt** (Numeric : SURVPTS):
  Ycoordinate of survey point
  Default=YPT
  Required=No

* **zpt** (Numeric : SURVPTS):
  Z coordinate of survey point
  Default=ZPT
  Required=No

* **from** (Numeric : SAMPLE1):
  Along channel distance to sample start.
  Default=FROM
  Required=No

* **to** (Numeric : SAMPLE1):
  Along channel distance to sample end
  Default=TO
  Required=No

### Parameters:

* **extend**:
  Option to extend channel beyond last survey point: 0 - Terminate channel at the last
  survey point. 1 - Extend channel to maximum TO value in the input samples files.
  Range=1, 3
  Values=1,2,3
  Default=1
  Required=No

* **endpoint**:
  Option to include the X, Y, Z coordinates of the start and end of each sample in the
  desurveyed output file. 0 - Start and end coordinates are not included in OUT file. 1 -
  Fields XSTART, YSTART, ZSTART, XEND, YEND and ZEND are created in the OUT file.
  Range=1,4
  Values=1,2,3,4
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('channl3d')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute channl3d
print("Running channl3d...")
dm_cmd.channl3d(
    survpts_i=['t_SurfacePointsPt'],  # required
    samples_i=['t_assays'],  # required
    # out_o='t_channl3d_out',  # optional
    # chansmry_o='t_channl3d_summary',  # optional
    # errors_o=['t_channl3d_errors'],  # optional
    # bhid_f='BHID',  # optional
    # xpt_f='optional',  # optional
    # ypt_f='optional',  # optional
    # zpt_f='optional',  # optional
    # from_f='FROM',  # optional
    # to_f='TO',  # optional
    # extend_p=1,  # optional
    # endpoint_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("channl3d execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_channl3d_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")